In [7]:
import pprint
import radiate as rd
import numpy as np
import polars as pl  # type: ignore

rd.random.seed(123)

In [8]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [9]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alter(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(
        rd.Limit.score(0.01), rd.Limit.generations(500), rd.Limit.convergence(50, 0.001)
    )
)

result = engine.run(log=True)

2026-11-15T16:46:52.163446Z  INFO State Change: PreStart -> Running
2026-11-15T16:46:52.166595Z  INFO Epoch 1    | Score:   1.1919 | Time: 2.79ms
2026-11-15T16:46:52.167821Z  INFO Epoch 2    | Score:   0.7937 | Time: 3.96ms
2026-11-15T16:46:52.169113Z  INFO Epoch 3    | Score:   0.5836 | Time: 5.13ms
2026-11-15T16:46:52.170082Z  INFO Epoch 4    | Score:   0.2036 | Time: 6.03ms
2026-11-15T16:46:52.171120Z  INFO Epoch 5    | Score:   0.2036 | Time: 6.97ms
2026-11-15T16:46:52.171994Z  INFO Epoch 6    | Score:   0.2036 | Time: 7.83ms
2026-11-15T16:46:52.172827Z  INFO Epoch 7    | Score:   0.2024 | Time: 8.64ms
2026-11-15T16:46:52.173848Z  INFO Epoch 8    | Score:   0.2024 | Time: 9.63ms
2026-11-15T16:46:52.174754Z  INFO Epoch 9    | Score:   0.2024 | Time: 10.51ms
2026-11-15T16:46:52.175778Z  INFO Epoch 10   | Score:   0.2024 | Time: 11.51ms
2026-11-15T16:46:52.176641Z  INFO Epoch 11   | Score:   0.2024 | Time: 12.36ms
2026-11-15T16:46:52.177502Z  INFO Epoch 12   | Score:   0.2024 | Time: 

In [10]:
metrics = result.metrics
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",80.0,9313.0,80.284485,2.205945,4.866191,11.496284,80.0,100.0,116,null,null,null,null,null,null,114,1,"[""statistic""]"
"""step.evaluate.time""",0.000669,0.101681,0.000442,0.000602,3.6247e-7,0.0,8.3000e-8,0.005408,230,101680µs,442µs,602µs,0µs,5407µs,0µs,114,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,2300.0,20.0,0.0,0.0,0.0,20.0,20.0,115,null,null,null,null,null,null,114,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000001,0.000252,0.000002,0.000002,5.4942e-12,0.0,0.000001,0.000023,115,251µs,2µs,2µs,1µs,22µs,0µs,114,1,"[""selector"", ""time""]"
"""selector.boltzmann""",80.0,9200.0,80.0,0.0,0.0,0.0,80.0,80.0,115,null,null,null,null,null,null,114,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""crossover.blend.rate""",0.7,80.5,0.7,0.0,0.0,0.0,0.7,0.7,115,null,null,null,null,null,null,114,1,"[""alterer"", ""crossover"", … ""expr""]"
"""step.metric.time""",0.000008,0.001167,0.00001,0.000005,2.5604e-11,0.0,0.000007,0.00004,115,1166µs,10µs,5µs,7µs,39µs,0µs,114,1,"[""time"", ""step""]"
"""time""",0.000781,0.119513,0.001039,0.000657,4.3141e-7,0.0,0.000776,0.005877,115,119513µs,1039µs,656µs,776µs,5877µs,0µs,114,1,"[""time""]"


In [11]:
print(result.metrics.dashboard())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["pct.carryover"].to_dict())

[34 metrics, 3797 updates]
----- Metrics ----- (34 :: 3797) 
  carryover: 0.115  diversity: 0.908  unique_members: 92  unique_scores: 92  improvements: 115  iter_time(mean): 1.04ms

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt      
---------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.350      | 0.000      | 3.000      | 100    | 0.035        | 0.809      | 0.000      | 0.000     
genome.size                | dist   | 88.000     | 88.000     | 88.000     | 100    | 8.800        | 0.000      | 0.000      | 0.000     
scores                     | dist   | 4.414      | 0.036      | 63.551     | 100    | 0.441        | 10.830     | 4.238      | 22.202    


== Statistics ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total

In [12]:
result.duration()

datetime.timedelta(microseconds=119513)